# Membrane creation

In [ ]:
import itertools
import logging
import os
import random
from copy import deepcopy
from multiprocessing import Pool
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from shapely import to_wkt, wkt
from shapely.affinity import rotate, translate
from shapely.geometry import MultiPolygon, Point, Polygon, box
from shapely.ops import unary_union
from shapely.strtree import STRtree
from tqdm import tqdm

import cyanomembranes as cm

(OUT := Path("../output")).mkdir(exist_ok=True)

# Random generator
RNG = np.random.default_rng(seed=0)

# CPUS
N_PROCESSES = 10 #6

In [ ]:
def any_overlap(polygons):
    n = len(polygons)
    for i in range(n):
        for j in range(i+1, n):
            if polygons[i].intersects(polygons[j]):
                fig,ax = plt.subplots()
                ax.plot(*polygons[i].exterior.xy, lw=1, c="blue")
                ax.plot(*polygons[j].exterior.xy, lw=1, c="blue")
                plt.show()
                return True
    return False

# Create protein shadows

In [ ]:
PDB_FILES = [
    "1JB0-PSI-syn-cocc.pdb",
    "1NEK-SDH-Ecoli.pdb",
    "1OCO-cytoxidase-bov.trpdb",
    "1xl4-Kchannel-Pmagnetotacticum.trpdb",
    "3WU2-PSII-ThermosynVul.pdb",
    "4H13-cytb6f.trpdb",
]

DATA = Path("../data_cyano/")

proteins = cm.pdb_utils.process_proteins(DATA, OUT / "protein_shadows", PDB_FILES)


In [ ]:
fig, ax = plt.subplots(3, 2, figsize=(8, 8))
ax = ax.flatten()
for i, prot in enumerate(proteins):
    polygon = proteins[prot]["polygon"]
    ax[i].plot(*polygon[0].exterior.xy)
    ax[i].set_aspect("equal")
    ax[i].set_title(prot)
    ax[i].set_xlim(-150, 150)
    ax[i].set_ylim(-150, 150)
plt.tight_layout()
fig.show()

## Membranes no MD

In [ ]:
# ----------------------------------

# CONFIGURATION

# ----------------------------------



NUMBER_OF_PROTEINS = {
    "3WU2-PSII-ThermosynVul": [100, 200, 300, 400, 500, 600, 700, 800, 900],
    "4H13-cytb6f": [10, 20, 30, 40, 50, 60, 70, 80, 90, 100, 200, 300, 400, 500, 600, 700, 800, 900, 1100, 1400],
    "1JB0-PSI-syn-cocc": [100, 200, 300, 400, 500],
    "avg_membrane": [10, 20, 30, 40, 50, 60, 70, 80, 90, 100, 110, 110]
}

ANGLES = range(360)
N = 10

# -----------------------------------

# LOGGING

# -----------------------------------

logging.basicConfig(
    format="%(asctime)s | %(levelname)s | %(message)s", level=logging.INFO
)

# --------------------------------

# UTILITY FUNCTIONS

# --------------------------------


def ensure_output_dir(out_dir: Path) -> None:
    """Ensure output directory exists"""
    out_dir.mkdir(parents=True, exist_ok=True)


def generate_rotated_polygons(
    n: int, base_polygon: Polygon, angles: range
) -> list[Polygon]:
    """Generate n rotated copies of a base polygon"""
    return [rotate(base_polygon, random.choice(angles)) for _ in range(n)]


def generate_roated_polygons_membrane(n_cytb6f: int, angles: range) -> list[Polygon]:
    return (
        [
            rotate(proteins["1JB0-PSI-syn-cocc"]["polygon"][0], random.choice(ANGLES))
            for _ in range(int(n_cytb6f * 4.76))
        ]
        + [
            rotate(
                proteins["3WU2-PSII-ThermosynVul"]["polygon"][0],
                random.choice(ANGLES),
            )
            for _ in range(int(n_cytb6f * 1.1))
        ]
        + [
            rotate(proteins["4H13-cytb6f"]["polygon"][0], random.choice(ANGLES))
            for _ in range(n_cytb6f)
        ]
    )


def save_polygons_to_wkt(file_path: Path, polygons: list[Polygon]) -> None:
    """Save polygons as WKT to a file"""
    with file_path.open("w") as f:
        for poly in polygons:
            f.write(poly.wkt + "\n")


# --------------------------------

# MAIN WORKER FUNCTION

# --------------------------------


def process_one_case(args: tuple[str, int, int, str]) -> Path:
    polygon_key, nprot, i = args

    # Polygon source and output directory
    if polygon_key != "avg_membrane":
        base_polygon = proteins[polygon_key]["polygon"][0]
    out_dir = OUT / polygon_key
    ensure_output_dir(out_dir)

    file_path = Path(out_dir / f"polygons_{polygon_key}_{nprot}-{i}.wkt")

    if file_path.exists():
        logging.info(f"{file_path.name} already exists, skipping.")
        return file_path

    if polygon_key == "avg_membrane":
        polygons = generate_roated_polygons_membrane(nprot, ANGLES)
    else:
        polygons = generate_rotated_polygons(nprot, base_polygon, ANGLES)
    world, placed_polygons, not_placed, ghosts = cm.geo_utils.placement_routine(
        polygons,
        dimensions=[5000, 5000],
        step_size=30,
        rotation_angle=360,
        max_iter_placement=600,
    )

    extended_world = world + list(np.array(ghosts).ravel())
    save_polygons_to_wkt(file_path, extended_world)
    logging.info(f"Saved {len(extended_world)} polygons to {file_path}")

    return file_path


tasks = [
    (polygon_key, nprot, i)
    for polygon_key, nprot_list in NUMBER_OF_PROTEINS.items()
    for nprot in nprot_list
    for i in range(N)
]

with Pool(processes=N_PROCESSES) as pool:
    results = pool.map(process_one_case, tasks)

logging.info(
    f"Completed processing {len(results)} tasks"
    f"across {len(NUMBER_OF_PROTEINS)} polygon types"
)

In [ ]:
with open(OUT / "Test_membranes.txt", "w") as f:
    for i in tasks:
        polygon_key, nprot, idx = i
        print(f"Processing {polygon_key}-{nprot}-{idx}")
        out_dir = OUT / polygon_key
        file_path = Path(out_dir / f"polygons_{polygon_key}_{nprot}-{idx}.wkt")
        p = cm.geo_utils.readwkt(file_path)
        protein_number = len(p) / 9
        overlap = any_overlap(p)
        status = "WARNING" if (overlap is True) or (protein_number != nprot) else "OK"
        string_to_print = f"{polygon_key}_{nprot}-{idx} \n # in file: {protein_number} \t # expected: {nprot} \t Overlap: {overlap} \t Status: {status} \n \n"
        f.write(string_to_print)
        f.flush()

## Small crystalline arrays PSI

In [ ]:
NUMBER_OF_PROTEINS = {"avg_membrane": [40, 60, 80]} # FOR PSI crystal "avg_membrane": [40, 60, 70]
CDEGREE = {
    "1JB0-PSI-syn-cocc": [0, 0.2]}
#, "3WU2-PSII-ThermosynVul": [0, 0.2]}
SAVE_IN_SMALL = True
ANGLES = range(360)
N = 10
MV = 1.2
NCRYSTALS = 4
STOP_AFTER_CRYSTAL = False


def small_rejection_sampler(free_space):

    minx, miny, maxx, maxy = free_space.bounds

    while True:
        x = np.random.uniform(minx, maxx)
        y = np.random.uniform(miny, maxy)
        p = Point(x, y)
        if free_space.contains(p):
            return x, y


def generate_roated_polygons_membrane(
    n_cytb6f: int, angles: range, return_indices: bool = False
) -> list[Polygon]:
    n_psi = int(n_cytb6f * 4.76)
    n_psii = int(n_cytb6f * 1.1)
    polygons = (
        [
            rotate(proteins["1JB0-PSI-syn-cocc"]["polygon"][0], random.choice(ANGLES))
            for _ in range(n_psi)
        ]
        + [
            rotate(
                proteins["3WU2-PSII-ThermosynVul"]["polygon"][0],
                random.choice(ANGLES),
            )
            for _ in range(n_psii)
        ]
        + [
            rotate(proteins["4H13-cytb6f"]["polygon"][0], random.choice(ANGLES))
            for _ in range(n_cytb6f)
        ]
    )

    indices = {
        "1JB0-PSI-syn-cocc": range(n_psi),
        "3WU2-PSII-ThermosynVul": range(n_psi, n_psi + n_psii),
        "4H13-cytb6f": range(n_psi + n_psii, n_psi + n_psii + n_cytb6f),
    }

    if return_indices:
        return polygons, indices
    else:
        return polygons


def get_crystals(n_prot, pkey, cdegree):
    test_polygons, indices = generate_roated_polygons_membrane(
        n_prot, ANGLES, return_indices=True
    )

    N_CRYSTALS = NCRYSTALS
    SIZE = int(len(indices[pkey]) * cdegree)
    selected_polygons = random.sample(indices[pkey], N_CRYSTALS * SIZE)
    print(f"Polygons in Crystal: {len(selected_polygons)}")
    print(f"Polygons per Crystal: {SIZE}")
    seeds = np.array(test_polygons)[random.sample(selected_polygons, N_CRYSTALS)]

    mask = np.ones(len(test_polygons), dtype=bool)
    mask[selected_polygons] = False
    test_polygons = np.array(test_polygons)[mask]
    # print(len(test_polygons))

    crystals = [
        cm.geo_utils.make_random_crystal_2d(
            seed=seed,
            max_variation=MV,
            lattice_type="hexagonal",
            n=SIZE,
            regular=True,
            exclude_first=False,
        )
        for seed in seeds
    ]

    placed_crystals = []
    ghosts_list = []
    surface = box(0, 0, 5000, 5000)

    for crystal in crystals:
        ghosts_list_long = list(np.array(ghosts_list).ravel())
        all_crystals = unary_union(placed_crystals + ghosts_list_long)
        free_space = surface.difference(all_crystals)
        cnt = 0
        while True:
            x, y = small_rejection_sampler(free_space)
            dx = x - MultiPolygon(crystal).centroid.x
            dy = y - MultiPolygon(crystal).centroid.y
            candidate = [translate(p, xoff=dx, yoff=dy) for p in crystal]
            candidate_ghosts = [
                cm.geo_utils._create_ghost(p, (0, 5000)) for p in candidate
            ]
            candidate_ghosts_long = list(np.array(candidate_ghosts).ravel())
            all_placed_ghosts = list(np.array(ghosts_list).ravel())
            tree = STRtree(placed_crystals + all_placed_ghosts)
            tree2 = STRtree(candidate_ghosts_long)

            cnt += 1
            if (
                not tree.query(candidate).size > 0
                and not tree.query(candidate_ghosts_long).size > 0
                and not tree2.query(candidate).size > 0
            ):
                print("yes")
                placed_crystals.extend(candidate)
                ghosts_list.extend(candidate_ghosts)
                free_space = free_space.difference(MultiPolygon(candidate))
                break

            if cnt > 100:
                print("couldn't find a place")
                return None, None, None

    print("Finished")
    print(len(ghosts_list))
    return list(test_polygons), list(np.array(placed_crystals).ravel()), ghosts_list

    # print("Finished")
    # return list(test_polygons), list(translated_crystals.ravel()), list(ghosts)


def process_one_case(args: tuple[str, int, str, float, int]) -> Path:
    polygon_key, nprot, crystal_key, cdegree, i = args

    # Polygon source and output directory
    if SAVE_IN_SMALL:
        out_dir = OUT / f"{polygon_key}_{crystal_key}_crystal_small"
    else:
        out_dir = OUT / f"{polygon_key}_{crystal_key}_crystal"
    ensure_output_dir(out_dir)

    file_path = Path(
        out_dir
        / f"polygons_{polygon_key}_{nprot}_{crystal_key}_{MV}_{cdegree}_{NCRYSTALS}-{i}.wkt"
    )

    if file_path.exists():
        logging.info(f"{file_path.name} already exists, skipping.")
        return file_path

    if cdegree == 0:
        polygons = generate_roated_polygons_membrane(nprot, angles=ANGLES)

        world, placed_polygons, not_placed, ghosts = cm.geo_utils.placement_routine(
            polygons,
            dimensions=[5000, 5000],
            step_size=30,
            rotation_angle=360,
            max_iter_placement=50,
        )

    elif cdegree == -1000:
        polygons, indices = generate_roated_polygons_membrane(
            nprot, angles=ANGLES, return_indices=True
        )
        cytb6f_indices = indices["4H13-cytb6f"]
        polygons = list(np.array(polygons)[cytb6f_indices])
        world, placed_polygons, not_placed, ghosts = cm.geo_utils.placement_routine(
            polygons,
            dimensions=[5000, 5000],
            step_size=30,
            rotation_angle=360,
            max_iter_placement=50,
        )

    else:
        polygons = None
        translated_crystals = None
        ghosts_crystals = None

        while (
            polygons is None and translated_crystals is None and ghosts_crystals is None
        ):
            polygons, translated_crystals, ghosts_crystals = get_crystals(
                n_prot=nprot, pkey=crystal_key, cdegree=cdegree
            )

        if STOP_AFTER_CRYSTAL:
            world = translated_crystals
            ghosts = ghosts_crystals

        else:
            world, placed_polygons, not_placed, ghosts = cm.geo_utils.placement_routine(
                polygons,
                start_world=deepcopy(translated_crystals),
                start_ghosts=deepcopy(ghosts_crystals),
                dimensions=[5000, 5000],
                step_size=30,
                rotation_angle=360,
                max_iter_placement=50,
                max_iter_shuffle=0,
            )

    extended_world = world + list(np.array(ghosts).ravel())
    save_polygons_to_wkt(file_path, extended_world)
    logging.info(f"Saved {len(extended_world)} polygons to {file_path}")
    if cdegree in [-1000, 0]:
        return file_path
    return file_path, translated_crystals, ghosts_crystals


tasks = [
    (membrane_key, nprot, protein_key, cdegree, i)
    for membrane_key, nprot_list in NUMBER_OF_PROTEINS.items()
    for nprot in nprot_list
    for protein_key in CDEGREE.keys()  # iterate over all proteins that can crystallize
    for cdegree in CDEGREE.get(protein_key)
    for i in range(N)
]
with Pool(processes=N_PROCESSES) as pool:
    results = pool.map(process_one_case, tasks)

logging.info(
    f"Completed processing {len(results)} tasks"
    f"across {len(NUMBER_OF_PROTEINS)} polygon types"
)


## Small crystalline arrays PSII

In [ ]:
NUMBER_OF_PROTEINS = {"avg_membrane": [40, 60, 70]} 
CDEGREE = {
    "3WU2-PSII-ThermosynVul": [0, 0.2]}
SAVE_IN_SMALL = True
ANGLES = range(360)
N = 10
MV = 1.2
NCRYSTALS = 4
STOP_AFTER_CRYSTAL = False


def small_rejection_sampler(free_space):

    minx, miny, maxx, maxy = free_space.bounds

    while True:
        x = np.random.uniform(minx, maxx)
        y = np.random.uniform(miny, maxy)
        p = Point(x, y)
        if free_space.contains(p):
            return x, y


def generate_roated_polygons_membrane(
    n_cytb6f: int, angles: range, return_indices: bool = False
) -> list[Polygon]:
    n_psi = int(n_cytb6f * 4.76)
    n_psii = int(n_cytb6f * 1.1)
    polygons = (
        [
            rotate(proteins["1JB0-PSI-syn-cocc"]["polygon"][0], random.choice(ANGLES))
            for _ in range(n_psi)
        ]
        + [
            rotate(
                proteins["3WU2-PSII-ThermosynVul"]["polygon"][0],
                random.choice(ANGLES),
            )
            for _ in range(n_psii)
        ]
        + [
            rotate(proteins["4H13-cytb6f"]["polygon"][0], random.choice(ANGLES))
            for _ in range(n_cytb6f)
        ]
    )

    indices = {
        "1JB0-PSI-syn-cocc": range(n_psi),
        "3WU2-PSII-ThermosynVul": range(n_psi, n_psi + n_psii),
        "4H13-cytb6f": range(n_psi + n_psii, n_psi + n_psii + n_cytb6f),
    }

    if return_indices:
        return polygons, indices
    else:
        return polygons


def get_crystals(n_prot, pkey, cdegree):
    test_polygons, indices = generate_roated_polygons_membrane(
        n_prot, ANGLES, return_indices=True
    )

    N_CRYSTALS = NCRYSTALS
    SIZE = int(len(indices[pkey]) * cdegree)
    selected_polygons = random.sample(indices[pkey], N_CRYSTALS * SIZE)
    print(f"Polygons in Crystal: {len(selected_polygons)}")
    print(f"Polygons per Crystal: {SIZE}")
    seeds = np.array(test_polygons)[random.sample(selected_polygons, N_CRYSTALS)]

    mask = np.ones(len(test_polygons), dtype=bool)
    mask[selected_polygons] = False
    test_polygons = np.array(test_polygons)[mask]
    # print(len(test_polygons))

    crystals = [
        cm.geo_utils.make_random_crystal_2d(
            seed=seed,
            max_variation=MV,
            lattice_type="hexagonal",
            n=SIZE,
            regular=True,
            exclude_first=False,
        )
        for seed in seeds
    ]

    placed_crystals = []
    ghosts_list = []
    surface = box(0, 0, 5000, 5000)

    for crystal in crystals:
        ghosts_list_long = list(np.array(ghosts_list).ravel())
        all_crystals = unary_union(placed_crystals + ghosts_list_long)
        free_space = surface.difference(all_crystals)
        cnt = 0
        while True:
            x, y = small_rejection_sampler(free_space)
            dx = x - MultiPolygon(crystal).centroid.x
            dy = y - MultiPolygon(crystal).centroid.y
            candidate = [translate(p, xoff=dx, yoff=dy) for p in crystal]
            candidate_ghosts = [
                cm.geo_utils._create_ghost(p, (0, 5000)) for p in candidate
            ]
            candidate_ghosts_long = list(np.array(candidate_ghosts).ravel())
            all_placed_ghosts = list(np.array(ghosts_list).ravel())
            tree = STRtree(placed_crystals + all_placed_ghosts)
            tree2 = STRtree(candidate_ghosts_long)

            cnt += 1
            if (
                not tree.query(candidate).size > 0
                and not tree.query(candidate_ghosts_long).size > 0
                and not tree2.query(candidate).size > 0
            ):
                print("yes")
                placed_crystals.extend(candidate)
                ghosts_list.extend(candidate_ghosts)
                free_space = free_space.difference(MultiPolygon(candidate))
                break

            if cnt > 100:
                print("couldn't find a place")
                return None, None, None

    print("Finished")
    print(len(ghosts_list))
    return list(test_polygons), list(np.array(placed_crystals).ravel()), ghosts_list

    # print("Finished")
    # return list(test_polygons), list(translated_crystals.ravel()), list(ghosts)


def process_one_case(args: tuple[str, int, str, float, int]) -> Path:
    polygon_key, nprot, crystal_key, cdegree, i = args

    # Polygon source and output directory
    if SAVE_IN_SMALL:
        out_dir = OUT / f"{polygon_key}_{crystal_key}_crystal_small"
    else:
        out_dir = OUT / f"{polygon_key}_{crystal_key}_crystal"
    ensure_output_dir(out_dir)

    file_path = Path(
        out_dir
        / f"polygons_{polygon_key}_{nprot}_{crystal_key}_{MV}_{cdegree}_{NCRYSTALS}-{i}.wkt"
    )

    if file_path.exists():
        logging.info(f"{file_path.name} already exists, skipping.")
        return file_path

    if cdegree == 0:
        polygons = generate_roated_polygons_membrane(nprot, angles=ANGLES)

        world, placed_polygons, not_placed, ghosts = cm.geo_utils.placement_routine(
            polygons,
            dimensions=[5000, 5000],
            step_size=30,
            rotation_angle=360,
            max_iter_placement=50,
        )

    elif cdegree == -1000:
        polygons, indices = generate_roated_polygons_membrane(
            nprot, angles=ANGLES, return_indices=True
        )
        cytb6f_indices = indices["4H13-cytb6f"]
        polygons = list(np.array(polygons)[cytb6f_indices])
        world, placed_polygons, not_placed, ghosts = cm.geo_utils.placement_routine(
            polygons,
            dimensions=[5000, 5000],
            step_size=30,
            rotation_angle=360,
            max_iter_placement=50,
        )

    else:
        polygons = None
        translated_crystals = None
        ghosts_crystals = None

        while (
            polygons is None and translated_crystals is None and ghosts_crystals is None
        ):
            polygons, translated_crystals, ghosts_crystals = get_crystals(
                n_prot=nprot, pkey=crystal_key, cdegree=cdegree
            )

        if STOP_AFTER_CRYSTAL:
            world = translated_crystals
            ghosts = ghosts_crystals

        else:
            world, placed_polygons, not_placed, ghosts = cm.geo_utils.placement_routine(
                polygons,
                start_world=deepcopy(translated_crystals),
                start_ghosts=deepcopy(ghosts_crystals),
                dimensions=[5000, 5000],
                step_size=30,
                rotation_angle=360,
                max_iter_placement=50,
                max_iter_shuffle=0,
            )

    extended_world = world + list(np.array(ghosts).ravel())
    save_polygons_to_wkt(file_path, extended_world)
    logging.info(f"Saved {len(extended_world)} polygons to {file_path}")
    if cdegree in [-1000, 0]:
        return file_path
    return file_path, translated_crystals, ghosts_crystals


tasks = [
    (membrane_key, nprot, protein_key, cdegree, i)
    for membrane_key, nprot_list in NUMBER_OF_PROTEINS.items()
    for nprot in nprot_list
    for protein_key in CDEGREE.keys()  # iterate over all proteins that can crystallize
    for cdegree in CDEGREE.get(protein_key)
    for i in range(N)
]
with Pool(processes=N_PROCESSES) as pool:
    results = pool.map(process_one_case, tasks)

logging.info(
    f"Completed processing {len(results)} tasks"
    f"across {len(NUMBER_OF_PROTEINS)} polygon types"
)


In [ ]:
# with open(OUT / "Test_membranes2.txt", "w") as f:
#     for i in tasks:
#         polygon_key, nprot, crystal_key, cdegree, idx = i
#         print(f"Processing {polygon_key}-{nprot}-{crystal_key}-{cdegree}-{idx}")
#         out_dir = OUT / f"{polygon_key}_{crystal_key}_crystal"
#         file_path = Path(out_dir / f"polygons_{polygon_key}_{nprot}_{crystal_key}_{MV}_{cdegree}-{idx}.wkt" )
#         p = cm.geo_utils.readwkt(file_path)
#         protein_number = len(p) / 9
#         overlap = any_overlap(p)
#         status = "WARNING" if (overlap is True) or (protein_number != nprot) else "OK"
#         string_to_print = f"{polygon_key}_{nprot}-{idx} \n # in file: {protein_number} \t # expected: {nprot} \t Overlap: {overlap} \t Status: {status} \n \n"
#         f.write(string_to_print)
#         f.flush()

## Big crystalline arrays PSI

In [ ]:

NUMBER_OF_PROTEINS = {"avg_membrane": [40, 60, 80]} # FOR PSII Crystal {"avg_membrane": [40, 60, 70]}
CDEGREE = {
    "1JB0-PSI-syn-cocc": [0, 0.4, 0.8],
    #"3WU2-PSII-ThermosynVul": [0, 0.4, 0.8]

}
SAVE_IN_SMALL = False
ANGLES = range(360)
N = 10
MV = 1.2 
NCRYSTALS = 1
STOP_AFTER_CRYSTAL = False


tasks = [
    (membrane_key, nprot, protein_key, cdegree, i)
    for membrane_key, nprot_list in NUMBER_OF_PROTEINS.items()
    for nprot in nprot_list
    for protein_key in CDEGREE.keys()  # iterate over all proteins that can crystallize
    for cdegree in CDEGREE.get(protein_key)
    for i in range(N)
]
with Pool(processes=N_PROCESSES) as pool:
    results = pool.map(process_one_case, tasks)

logging.info(
    f"Completed processing {len(results)} tasks"
    f"across {len(NUMBER_OF_PROTEINS)} polygon types"
)


## Big crystalline arrays PSII

In [ ]:

NUMBER_OF_PROTEINS = {"avg_membrane": [40, 60, 70]}
CDEGREE = {
    "3WU2-PSII-ThermosynVul": [0, 0.4, 0.8]

}
SAVE_IN_SMALL = False
ANGLES = range(360)
N = 10
MV = 1.2 
NCRYSTALS = 1
STOP_AFTER_CRYSTAL = False


tasks = [
    (membrane_key, nprot, protein_key, cdegree, i)
    for membrane_key, nprot_list in NUMBER_OF_PROTEINS.items()
    for nprot in nprot_list
    for protein_key in CDEGREE.keys()  # iterate over all proteins that can crystallize
    for cdegree in CDEGREE.get(protein_key)
    for i in range(N)
]
with Pool(processes=N_PROCESSES) as pool:
    results = pool.map(process_one_case, tasks)

logging.info(
    f"Completed processing {len(results)} tasks"
    f"across {len(NUMBER_OF_PROTEINS)} polygon types"
)


# Make pure crystals

In [ ]:
# --------------------------------------------
# CONFIGURE
# --------------------------------------------

PROTEIN_KEYS = ["1JB0-PSI-syn-cocc", "3WU2-PSII-ThermosynVul"]
LATTICE_TYPES = ["square", "hexagonal"]
MAX_VARIATIONS = [1.0, 1.2, 1.4, 1.8, 2.0]
REPLICATES = 10
N_CELLS = 1000
REGULAR = True
EXCLUDE_FIRST = False

# --------------------------------------------
# LOGGING
# --------------------------------------------

logging.basicConfig(
    format="%(asctime)s | %(levelname)s | %(message)s", level=logging.INFO
)
logger = logging.getLogger(__name__)


# --------------------------------------------
# WORKER FUNCTION
# --------------------------------------------


def create_one_crystal(task):
    protein_key, lattice_type, max_variation, idx = task

    out_dir = OUT / f"{protein_key}_crystal"
    ensure_output_dir(out_dir)

    str_max_variation = str(round(max_variation, 2)).replace(".", "-")
    file_path = (
        out_dir
        / f"crystal_{protein_key}_{lattice_type}_regular{REGULAR}_{str_max_variation}_{idx}.wkt"
    )

    if file_path.exists():
        logger.info(f"Skipping {file_path.name}, already exists.")
        return file_path

    seed = rotate(proteins[protein_key]["polygon"][0], random.choice(ANGLES))
    crystal = cm.geo_utils.make_random_crystal_2d(
        seed=seed,
        max_variation=max_variation,
        lattice_type=lattice_type,
        n=N_CELLS,
        regular=REGULAR,
        exclude_first=EXCLUDE_FIRST
    )

    with file_path.open("w") as f:
        for poly in crystal:
            f.write(poly.wkt + "\n")

    logger.info(f"Saved crystal: {file_path.name} ({len(crystal)} polygons)")
    return file_path


tasks = list(
    itertools.product(PROTEIN_KEYS, LATTICE_TYPES, MAX_VARIATIONS, range(REPLICATES))
)

total_tasks = len(tasks)
logger.info("Start crystal generation")

with Pool(processes=N_PROCESSES) as pool:
    results = list(tqdm(pool.imap(create_one_crystal, tasks), total=total_tasks))

logger.info("Finished generating crystals")